In [0]:
from datetime import datetime, timedelta

# ============================================================
# CONFIG
# ============================================================

# START_END_DATE = "2025-08-08"
# END_END_DATE   = "2026-03-06"
# STEP_DAYS      = 7

DB_TEMP = "ccex_tmp"

# Source tables
T_MAU   = "ccex_tmp.okr_segment_tags_final_base" # new base table
T_NUF   = "ccex.ccex_nuf_weekly_temp"  # first_event_ts source (NEW/NUV)
T_TRON  = "ccex.ccex_tron_activity_props"
T_SPARK = "spark.spark_tron_activity_props"
T_AU_Temp = "ccex.ccex_au_temp"

# Temp tables (overwritten per end_date)
T_BASE           = f"{DB_TEMP}.li_mau_base_dedup_tmp"
T_FIRST_TS       = f"{DB_TEMP}.li_first_ts_tmp"
T_ACTIVITY       = f"{DB_TEMP}.li_activity_tmp"
T_EXPORT_ACT     = f"{DB_TEMP}.li_export_activity_tmp"
T_KPI_EDITOR     = f"{DB_TEMP}.li_kpi_editor_tmp"
T_KPI_EXPORT     = f"{DB_TEMP}.li_kpi_export_tmp"
T_KPI_RETURNS    = f"{DB_TEMP}.li_kpi_returns_tmp"
T_KPI_ENGAGEMENT = f"{DB_TEMP}.li_kpi_engagement_tmp"
T_KPI_RWAU       = f"{DB_TEMP}.li_kpi_rwau_tmp"
T_USER_LEVEL     = f"{DB_TEMP}.li_user_level_tmp"
T_NUV_USER_LEVEL = f"{DB_TEMP}.li_nuv_user_level_tmp"

# Legacy table name kept only if older cells still reference it
T_ENGAGED_MAU    = f"{DB_TEMP}.li_engaged_mau_tmp"

# Output table 
T_OUT_METRICS = f"{DB_TEMP}.core_leading_indicator_metrics_mau_v2"

#### events tables
ccex_tron_activity_props_DB = "ccex"
ccex_tron_activity_props_tbl = "ccex_tron_activity_props"
ccex_event_activity_props_tbl = "ccex_event_activity_b"

spark_tron_activity_props_DB = "spark"
spark_tron_activity_props_tbl = "spark_tron_activity_props"


# ============================================================
# NON CONSENT CONFIG — Add new tags here as needed!
# ============================================================
NON_CONSENT_TAGS = [
    'non-consent',
    'standalone',
    '3P FPL non-consent',
    '3P Integrations'
]

# Non consent table
T_PRECONSENT = "ccex.ccex_tron_preconsent_deduped_mau"

# Build SQL IN list dynamically
NON_CONSENT_TAGS_SQL = ", ".join(
    [f'"{tag}"' for tag in NON_CONSENT_TAGS]
)

#-------------------------------------------------------------------------------

# -------------------------
# Segment logic
# -------------------------

CORE_FILTER_SQL = """
source_name IN ('CCEX','1.N')
AND lower(coalesce(platform_category,"")) != 'dapp-windows'
AND lower(coalesce(offer_category,"")) NOT LIKE '%edu%'
"""

CC_FILTER_SQL = """m.cloud_type_value = 'Creative Cloud Subs'
        AND b.market_segment_category = 'Adobe Members Entitled with Express Premium'
        AND lower(coalesce(b.contract_type,"")) <> 'edu enterprise k12'
        AND lower(coalesce(b.offer_type,"")) <> 'acrobat sa'
        """

third_party_filter_sql = """ lower(channel_detail) like '%third party%' """

ACROBAT_PHOTOS_SUBSEGMENTS = ["Acrobat - Photos"]
ACROBAT_NONPHOTOS_SUBSEGMENTS = ["Acrobat - Non Photos"]

# New requested segment mappings
EXPRESS_PHOTOS_SUBSEGMENTS = ["Express - Photos"]
STANDALONE_EXCL_EMBED_EDU_ENTITLED_SUBSEGMENTS = ["Product"]

cc_entitled = ['CC Entitled']
third_party = [
    '3P Integrations',
    '3P Telecom Companies: Airtel India'
]
edu = ['Edu']

SEGMENTS = [
    {"name": "OVERALL",            "where": "1=1"},
    {"name": "Acrobat photos",     "where": "acrobat_photos_flag = 1"},
    {"name": "Acrobat non-photos", "where": "acrobat_nonphotos_flag = 1"},
    {"name": "Express - Photos",   "where": "express_photos_flag = 1"},
    {"name": "Standalone(excl. Embed, excl. Edu Entitled)", "where": "standalone_excl_embed_edu_entitled_flag = 1"},
    {"name": "edu",                "where": "edu_flag=1"},
    {"name": "cc entitled",        "where": "cc_entitled_flag=1"},
    {"name": "3p",                 "where": "third_party_flag=1"}
]

# ============================================================
# Helpers 
# ============================================================

def exec_sql(sql: str):
    spark.sql(sql)

def daterange(start_date: str, end_date: str, step_days: int):
    s = datetime.strptime(start_date, "%Y-%m-%d").date()
    e = datetime.strptime(end_date, "%Y-%m-%d").date()
    d = s
    while d <= e:
        yield d.strftime("%Y-%m-%d")
        d += timedelta(days=step_days)


def ensure_output_tables():
    exec_sql(f"""
    CREATE TABLE IF NOT EXISTS 
    {T_OUT_METRICS} (

        -- ======================
        -- Dimensions
        -- ======================
        platform_cut      STRING,
        new_or_return     STRING,
        market_area       STRING,
        auth_status       STRING,
        segment           STRING,

        -- ======================
        -- Base volumes
        -- ======================
        MAU               BIGINT,
        pnuv_users        BIGINT,
        nuv_users         BIGINT,
        total_visitors    BIGINT,

        -- ======================
        -- FTA / Auth
        -- ======================
        d1_fta_users      BIGINT,
        d1_fta_pct        DOUBLE,
        fta_users         BIGINT,
        fta_pct           DOUBLE,
      
        -- ======================
        -- Editor / Engagement
        -- ======================
        d1_editor_users   BIGINT,
        d1_editor_pct     DOUBLE,

        d1_engaged_users  BIGINT,
        d1_engaged_pct    DOUBLE,

        editor_users      BIGINT,
        editor_pct        DOUBLE,

        -- ======================
        -- Export
        -- ======================
        d1_export_users   BIGINT,
        d1_export_pct     DOUBLE,

        export_users      BIGINT,
        export_pct        DOUBLE,

        -- ======================
        -- Short-term return
        -- ======================
        d2_7_users        BIGINT,
        d2_7_pct          DOUBLE,

        -- ======================
        -- Week-1 retention (new-user WAU based)
        -- ======================
        w1_rr_users       BIGINT,
        w1_rr_pct         DOUBLE,

        -- ======================
        -- Month-1 retention
        -- ======================
        m1_rr_users       BIGINT,
        m1_rr_pct         DOUBLE,

        -- ======================
        -- All-user rWAU retention
        -- ======================
        rwau_users        BIGINT,
        rwau_pct          DOUBLE,

        -- ======================
        -- rMAU retention
        -- ======================
        rmau_users        BIGINT,
        rmau_pct          DOUBLE,

        -- ======================
        -- Engaged MAU
        -- ======================
        engaged_mau       BIGINT,
        engaged_mau_pct   DOUBLE
    )
    PARTITIONED BY (end_date DATE)
    """)

ensure_output_tables()

In [0]:
# DB_TEMP="ccex_tmp"

# =========================================================
# 1) STATIC TEMP VIEW(S)
# =========================================================
def create_market_segment_view():
    market_seg_df = (
        spark.table("spark.market_segment_categorization")
        .selectExpr(
            "offer_category_key",
            "sku_code_primary",
            "offer_category",
            "offer_subcategory",
            "contract_type",
            "sku_market_segment_primary",
            "cloud_type",
            "sku_cc_segment_primary as cc_segment",
            "explode(market_segment_category) as market_segment_category",
            "offer_type",
            "dme_acct_segment"
        )
        .distinct()
    )
    market_seg_df.createOrReplaceTempView("ms_categorization")

create_market_segment_view()

# =========================================================
# 2) Acrobat / Airtel TEMP VIEW(S)
# =========================================================

airtel_events = """ 
      -- airtel
      'access-from-deeplink',
      'access-app-complete',
      'open-editor','authentication:loginSucceeded','project:editorDisplayed',
      'view-express-home','homePageViewed',
      'view-quickaction-upload-page','quickAction:uploadPageViewed','pageload-complete',
      'initialize-app-launch','open-published-link','view-community-wall',
      'select-cover-page-service-start'
"""

airtel = '''
-- Airtel flag as 0/1 int 
  MAX(
    CASE
      WHEN e.user_properties['custom.user.offer_id'] = 'DE3F04C37860C5E1DA0BD640AF1BF621'
        OR e.event_properties['custom.link.id'] = 'partnerpath'
      THEN 1 ELSE 0
    END
  ) AS airtel_offer_id
'''

acrobat_events = """   
      'start-import-media',
      'open-editor','view-quickaction-upload-page','select-quickaction-asset','view-community-wall',
      'pageload-complete','access-app-complete',
      'access-from-deeplink',
      'initialize-app-launch',
      'copy-link-url','select-template','generate-presentation-complete','invite-sent',
      'editor-asset-load-complete'
"""

acrobat_flags = '''/* ---------- Acrobat/Integration flags ---------- */
  MAX(CASE WHEN 
             e.event_name in ( 'open-editor','pageload-complete','access-app-complete')
             and  e.event_properties['custom.sdk.client_name'] = 'Adobe Acrobat Extension' THEN 1 ELSE 0 END) = 1
    AS is_acrobat_extension_user,

  MAX(CASE 
         WHEN 
             e.event_name = 'initialize-app-launch'
            AND e.user_properties['hz.source_platform_type'] = 'desktop-app'
            AND e.user_properties['custom.user.installation_source'] <> 'ccd'
           THEN 1 ELSE 0 END) = 1
    AS is_adobe_express_photos_user,

  MAX(CASE WHEN e.event_name IN ('access-app-complete','pageload-complete','open-editor','invite-sent','copy-link-url','select-template','generate-presentation-complete')
            AND e.event_properties['custom.sdk.workflow_intent'] = 'edit-generative-presentation'
           THEN 1 ELSE 0 END) = 1
    AS is_generative_reuse_user,

  MAX(CASE WHEN e.event_properties['custom.sdk.client_id'] = 'ReaderMobileAndroid4_0002'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%create%'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%recent%'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%na%'
           THEN 1 ELSE 0 END) = 1
    AS is_acrobat_mobile_edit_image_user,

  MAX(CASE WHEN (
            (
              e.event_properties['custom.sdk.client_id'] IN (
                'AdobeReader9','AdobeReader9RCM','AdobeReader9PPTRCM','AdobeReader9OutlookRCM',
                'AdobeAcrobat9','AdobeAcrobat9PPTRCM','AdobeAcrobat9RCM','AdobeAcrobat9OutlookRCM','dc-prod-virgoweb'
              )
              AND e.event_properties['custom.sdk.workflow_intent'] IN (
                'image-module-v2','text-to-image-module','text-to-image-module-v2','search-and-generate-workflow','image-module'
              )
            )
            OR
            (
              e.event_name = 'start-import-media'
              AND e.event_properties['custom.content.upload_method'] = 'transform-upload'
              AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%acrobat%'
            )
          ) THEN 1 ELSE 0 END) = 1
    AS is_enhanced_pdf_editing_user,

  MAX(CASE WHEN (
            (
              e.event_properties['custom.sdk.client_id'] = 'ReaderMobileAndroid4_0002'
              AND (
                lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%create%'
                OR lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%recent%'
                OR lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%na%'
              )
            )
            OR
            (
              e.event_name IN ('access-app-complete','pageload-complete','open-editor','select-template')
              AND e.event_properties['custom.referrer.app'] IN ('acrobat-web-studio','acrobat-desktop-studio','acrobat-reader-studio')
            )
            OR
            (
              e.event_name IN ('access-app-complete','pageload-complete','open-editor')
              AND e.event_properties['custom.sdk.client_cta_location'] IN (
                'create','studio-home','suggested-tools','all-tools','create-menu',
                'prompt-bar-generate','prompt-bar-tools','create-menu-dropdown','create-tab-card','all-tools-create-section-card'
              )
              AND e.event_properties['custom.sdk.client_id'] IN ('AdobeAcrobat9','dc-prod-virgoweb')
            )
            OR
            (e.event_name = 'start-import-media' AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%studio%')
            OR
            (e.event_name = 'start-import-media' AND lower(coalesce(e.event_properties['custom.referrer.app_intent'],'')) LIKE '%studio%')
          ) THEN 1 ELSE 0 END) = 1
    AS is_acrobat_original_creation_user,

  MAX(CASE WHEN (
      (e.event_name = 'start-import-media'
        AND e.event_properties['custom.content.upload_method'] = 'transform-upload'
        AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%acrobat%')
      OR
      (e.event_name IN ('open-editor','view-quickaction-upload-page','select-quickaction-asset','view-community-wall')
        AND e.event_properties['custom.sdk.client_name'] IN ('Acrobat','Adobe Acrobat Extension'))
      OR
      (e.event_name IN ('pageload-complete','access-app-complete')
        AND e.event_properties['custom.sdk.client_name'] IN ('Acrobat','Adobe Acrobat Extension')
        AND e.auth_flag = 'true')
      OR
      (e.event_name = 'access-from-deeplink'
        AND e.event_properties['custom.link.full'] IN (
          'https://adobesparkpost.app.link/83JRwB0j9Qb',
          'https://adobesparkpost.app.link/n7T79GOk9Qb',
          'https://adobesparkpost.app.link/mRjdnB4mDRb',
          'https://adobesparkpost.app.link/f54E0t01ZLb',
          'https://adobesparkpost.app.link/mBBoTGm2ZLb',
          'https://adobesparkpost.app.link/c8iaL3disSb',
          'https://adobesparkpost.app.link/oJcM4qKHsSb'
        )
        AND e.auth_flag = 'true')
      OR
      (e.event_name = 'initialize-app-launch'
        AND e.user_properties['hz.source_platform_type'] = 'desktop-app'
        AND e.user_properties['custom.user.installation_source'] <> 'ccd')
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor','copy-link-url','select-template','generate-presentation-complete')
        AND e.event_properties['custom.sdk.workflow_intent'] = 'edit-generative-presentation')
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor')
        AND e.event_properties['custom.referrer.app'] IN ('acrobat-web-studio','acrobat-desktop-studio'))
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor')
        AND e.event_properties['custom.sdk.client_cta_location'] IN (
          'create','studio-home','suggested-tools','all-tools','create-menu',
          'prompt-bar-generate','prompt-bar-tools','create-menu-dropdown','create-tab-card'
        )
        AND e.event_properties['custom.sdk.client_id'] IN ('AdobeAcrobat9','dc-prod-virgoweb'))
      OR
      (e.event_name = 'start-import-media'
        AND lower(coalesce(e.event_properties['custom.referrer.app_intent'],'')) LIKE '%studio%')
    ) THEN 1 ELSE 0 END) = 1 AS is_acrobat_integration_user,

  -- non consent Acrobat
  MAX(CASE WHEN e.event_name in ('editor-asset-load-complete') THEN 1 ELSE 0 END)=1 AS is_acrobat_non_consent
'''

def build_event_activity_view(end_date: str, period=28):
    event_activity_sql = f"""
    WITH auth AS (
      SELECT
        event_user_guid,
        max(e.user_properties['hz.source_platform_type']) as source_platform_type,
        max(e.user_properties['custom.user.installation_source']) as installation_source,
        {acrobat_flags},
        {airtel}
      FROM {ccex_tron_activity_props_DB}.{ccex_event_activity_props_tbl} e
      WHERE event_date BETWEEN DATE_SUB('{end_date}', {period}-1) AND '{end_date}'
        AND auth_flag = true
        AND event_name IN ({acrobat_events}, {airtel_events})
      GROUP BY ALL
    ),

    unauth AS (
      SELECT
        event_visitor_guid as event_user_guid,
        max(e.user_properties['hz.source_platform_type']) as source_platform_type,
        max(e.user_properties['custom.user.installation_source']) as installation_source,
        {acrobat_flags},
        {airtel}
      FROM {ccex_tron_activity_props_DB}.{ccex_event_activity_props_tbl} e
      WHERE event_date BETWEEN DATE_SUB('{end_date}', {period}-1) AND '{end_date}'
        AND auth_flag = false
        AND event_name IN ({acrobat_events}, {airtel_events})
      GROUP BY ALL
    )

    SELECT * FROM auth
    UNION ALL
    SELECT * FROM unauth
    """
    spark.sql(event_activity_sql).createOrReplaceTempView("event_activity_on_date_vw")

In [0]:
# ============================================================
# 1) Base: dedupe exploded MAU table
# ============================================================

def build_base_dedup(end_date: str):
    photos_in = ", ".join([f"'{x}'" for x in ACROBAT_PHOTOS_SUBSEGMENTS]) or "''"
    nonphotos_in = ", ".join([f"'{x}'" for x in ACROBAT_NONPHOTOS_SUBSEGMENTS]) or "''"
    express_photos_in = ", ".join([f"'{x}'" for x in EXPRESS_PHOTOS_SUBSEGMENTS]) or "''"
    standalone_excl_embed_edu_entitled_in = ", ".join(
        [f"'{x}'" for x in STANDALONE_EXCL_EMBED_EDU_ENTITLED_SUBSEGMENTS]
    ) or "''"
    edu_in = ", ".join([f"'{x}'" for x in edu]) or "''"
    cc_entitled_in = ", ".join([f"'{x}'" for x in cc_entitled]) or "''"
    third_party_in = ", ".join([f"'{x}'" for x in third_party]) or "''"

    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_BASE} AS
    SELECT
      date_date AS end_date,
      user_guid,
      platform_category AS platform_cut,
      new_or_return,
      market_area,

      -- pNUV membership
      MAX(CASE WHEN new_or_return IN ('NEW','NUV')
          THEN 1 ELSE 0 END) AS pnuv_flag,

      -- T4 return flags
      MAX(CASE WHEN TRIM(COALESCE(nuv_active_t4_friday,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_t4_return_flag,
      MAX(CASE WHEN TRIM(COALESCE(mau_active_t4_friday,'')) <> ''
          THEN 1 ELSE 0 END) AS mau_t4_return_flag,

      -- Photos specific flags
      MAX(CASE WHEN TRIM(COALESCE(nuv_users_segment,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_photos_flag,
      MAX(CASE WHEN TRIM(COALESCE(nuv_active_t4_friday_segment,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_photos_t4_return_flag,
      MAX(new_or_return_photos) AS new_or_return_photos,

      -- Segment flags
      MAX(CASE WHEN ({CORE_FILTER_SQL})
          THEN 1 ELSE 0 END) AS core_flag,

      MAX(CASE WHEN segment_tag IN ({photos_in})
          THEN 1 ELSE 0 END) AS acrobat_photos_flag,
      MAX(CASE WHEN segment_tag IN ({nonphotos_in})
          THEN 1 ELSE 0 END) AS acrobat_nonphotos_flag,

      -- New requested segment flags
      MAX(CASE WHEN segment_tag IN ({express_photos_in})
          THEN 1 ELSE 0 END) AS express_photos_flag,
      MAX(CASE WHEN segment_tag IN ({standalone_excl_embed_edu_entitled_in})
          THEN 1 ELSE 0 END) AS standalone_excl_embed_edu_entitled_flag,

      MAX(CASE WHEN segment_tag IN ({edu_in})
          THEN 1 ELSE 0 END) AS edu_flag,
      MAX(CASE WHEN segment_tag IN ({cc_entitled_in})
          THEN 1 ELSE 0 END) AS cc_entitled_flag,
      MAX(CASE WHEN segment_tag IN ({third_party_in})
          THEN 1 ELSE 0 END) AS third_party_flag

    FROM {T_MAU} m
    LEFT ANTI JOIN (
        SELECT * FROM {T_PRECONSENT}
        WHERE date_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    ) pc
      ON m.user_guid = pc.visitor_guid
      AND lower(m.segment_tag) IN ({NON_CONSENT_TAGS_SQL})

    WHERE m.date_date = '{end_date}'
    GROUP BY ALL
    """)


# ============================================================
# 2) Join first_event_ts + auth_flag for pNUV users
# ============================================================

def build_first_ts(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_FIRST_TS} AS
    WITH nuf AS (
      SELECT
        user_guid,
        MAX(auth_flag) AS auth_flag,
        MIN(event_ts)  AS first_event_ts
      FROM {T_NUF}
      WHERE date_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      GROUP BY user_guid
    ),

    nuf_wau AS (
      SELECT DISTINCT
        a.user_guid,
        1 AS pnuv_wau
      FROM {T_NUF} a
      INNER JOIN {T_AU_Temp} b
        ON a.user_guid = b.user_guid
      WHERE a.date_date = '{end_date}'
    )

    SELECT
      b.*,
      CASE WHEN b.pnuv_flag = 1 THEN n.first_event_ts ELSE NULL END AS first_event_ts,
      CASE WHEN b.pnuv_flag = 1 THEN n.auth_flag ELSE NULL END AS auth_flag,
      NVL(nw.pnuv_wau,0) AS pnuv_wau
    FROM {T_BASE} b
    LEFT JOIN nuf n
      ON b.user_guid = n.user_guid
    LEFT JOIN nuf_wau nw
      ON b.user_guid = nw.user_guid
    """)


# ============================================================
# 3) Activity union
# ============================================================

def build_activity(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_ACTIVITY} AS

    -- CCEX auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date,
      platform        AS ea_platform,
      auth_flag       AS ea_auth_flag,
      event_name,
      source_name     AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND source_name IN ('ESDK','CCEX')
      AND auth_flag = true
      AND (
            event_name IN ('access-app-complete','view-express-home','open-editor',
                           'view-quickaction-upload-page','view-quickaction-editor')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR (event_name = 'initialize-app-launch' AND event_properties['source.client_id'] = 'harmony-win-service')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.first_visit_page_url']) LIKE '%/template%'
             AND event_date >= '2026-01-31')
      )

    UNION ALL

    -- CCEX auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date,
      platform           AS ea_platform,
      auth_flag          AS ea_auth_flag,
      event_name,
      source_name        AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND source_name IN ('ESDK','CCEX')
      AND auth_flag = false
      AND (
            event_name IN ('access-app-complete','view-express-home','open-editor',
                           'view-quickaction-upload-page','view-quickaction-editor')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR (event_name = 'initialize-app-launch' AND event_properties['source.client_id'] = 'harmony-win-service')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.first_visit_page_url']) LIKE '%/template%'
             AND event_date >= '2026-01-31')
      )

    UNION ALL

    -- 1.N auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date,
      platform        AS ea_platform,
      auth_flag       AS ea_auth_flag,
      event_name,
      source_name     AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND auth_flag = true
      AND event_name IN ('experiment:standard:unauthenticated:assigned','authenticated:unknownEventName')
      AND (
            (LOWER(platform)='web' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR LOWER(platform)='mobile'
      )

    UNION ALL

    -- 1.N auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date,
      platform           AS ea_platform,
      auth_flag          AS ea_auth_flag,
      event_name,
      source_name        AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND auth_flag = false
      AND event_name IN ('experiment:standard:unauthenticated:assigned','authenticated:unknownEventName')
      AND (
            (LOWER(platform)='web' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR LOWER(platform)='mobile'
      )
    """)


# ============================================================
# 4) Export activity union
# ============================================================

def build_export_activity(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_EXPORT_ACT} AS

    -- CCEX auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = true
      AND (
        (
          event_date < '2023-09-06'
          AND event_name IN (
              'export-project-complete',
              'select-composer-publish-now-option',
              'select-post-composer-schedule',
              'export-quickaction-complete',
              'create-template',
              'publish-embed-project-complete'
          )
        )
        OR
        (
          event_date >= '2023-09-06'
          AND event_name IN ('export-project-complete','publish-embed-project-complete')
        )
      )

    UNION ALL

    -- CCEX auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = false
      AND event_name IN (
        'export-project-complete-unauth',
        'export-project-complete',
        'publish-embed-project-complete',
        'select-composer-publish-now-option',
        'select-post-composer-schedule',
        'export-quickaction-complete'
      )

    UNION ALL

    -- 1.N auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = true
      AND event_name IN ('libraries:templateCreated','project:exportCompleted','contentCal:postScheduled','contentCal:postPublished')

    UNION ALL

    -- 1.N auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = false
      AND event_name IN ('libraries:templateCreated','project:exportCompleted','contentCal:postScheduled','contentCal:postPublished')
    """)


# ============================================================
# 5) KPI: D1 editor open + total editor open
# ============================================================

def build_kpi_editor(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_EDITOR} AS
    WITH b2 AS (
      SELECT
        user_guid,
        pnuv_flag,
        COALESCE(
          try_to_timestamp(first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AS first_ts
      FROM {T_FIRST_TS}
    ),

    a2 AS (
      SELECT
        ea_user_guid,
        COALESCE(
          try_to_timestamp(ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AS ea_ts,
        event_name
      FROM {T_ACTIVITY}
    )

    SELECT
      b2.user_guid,

      MAX(CASE
        WHEN b2.pnuv_flag = 1
         AND b2.first_ts IS NOT NULL
         AND a2.ea_ts IS NOT NULL
         AND a2.ea_ts BETWEEN b2.first_ts AND b2.first_ts + INTERVAL 1 DAY
         AND a2.event_name IN ('open-editor','view-quickaction-upload-page','view-quickaction-editor')
        THEN 1 ELSE 0
      END) AS d1_editor_open,

      MAX(CASE
        WHEN a2.ea_ts IS NOT NULL
         AND to_date(a2.ea_ts) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
         AND a2.event_name IN ('open-editor','view-quickaction-upload-page','view-quickaction-editor')
        THEN 1 ELSE 0
      END) AS total_editor_open

    FROM b2
    LEFT JOIN a2
      ON b2.user_guid = a2.ea_user_guid
    GROUP BY b2.user_guid
    """)


# ============================================================
# 6) KPI: D1 export + total export
# ============================================================

def build_kpi_export(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_EXPORT} AS
    SELECT
      b.user_guid,

      MAX(CASE
        WHEN b.pnuv_flag = 1
         AND COALESCE(
              try_to_timestamp(e.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(e.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) BETWEEN COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             )
             AND COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) + INTERVAL 1 DAY
        THEN 1 ELSE 0
      END) AS d1_export_cnt,

      MAX(CASE
        WHEN to_date(e.ea_event_ts) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
        THEN 1 ELSE 0
      END) AS total_export_cnt

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_EXPORT_ACT} e
      ON b.user_guid = e.ea_user_guid
    GROUP BY b.user_guid
    """)


# ============================================================
# 7) KPI: Return flags D2-7, D1 Auth, W1
# ============================================================

def build_kpi_returns(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_RETURNS} AS
    SELECT
      b.user_guid,

      MAX(CASE
        WHEN b.pnuv_flag = 1
         AND COALESCE(
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) >= COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) + INTERVAL 1 DAY
         AND COALESCE(
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) <= COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) + INTERVAL 7 DAY
        THEN 1 ELSE 0
      END) AS d2_7_return_flag,

      MAX(CASE
        WHEN lower(coalesce(a.ea_auth_flag,'')) = 'true'
         AND to_date(a.ea_event_ts) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
        THEN 1 ELSE 0
      END) AS auth_flag,

      MAX(CASE
        WHEN lower(coalesce(a.ea_auth_flag,'')) = 'true'
         AND COALESCE(
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) BETWEEN COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             )
             AND COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) + INTERVAL 1 DAY
        THEN 1 ELSE 0
      END) AS d1_auth_flag,

      MAX(CASE
        WHEN b.pnuv_flag = 1
         AND b.pnuv_wau = 1
         AND to_date(a.ea_event_ts) BETWEEN date_add(to_date('{end_date}'),1) AND date_add(to_date('{end_date}'),7)
        THEN 1 ELSE 0
      END) AS w1_return_flag

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_ACTIVITY} a
      ON b.user_guid = a.ea_user_guid
    GROUP BY b.user_guid
    """)


# ============================================================
# 8) KPI: D1 Engaged + Engaged MAU
# ============================================================

ENGAGED_EVENT_CONDITION_SQL = """
    lower(event_name) IN (
        'add-asset','add-page','add-text','adjust-animation','adjust-border',
        'adjust-chart-settings','adjust-edge','adjust-effect','adjust-grid',
        'adjust-opacity','adjust-shape-effect','adjust-text-outline-thickness',
        'adjust-video-speed','apply-adjustments','apply-animation',
        'apply-auto-caption-color','apply-auto-caption-color-outline',
        'apply-auto-caption-shape-color','apply-auto-caption-shape-color-shadow',
        'apply-auto-caption-style','apply-auto-enhance','apply-ai-style',
        'apply-blend-mode','apply-border-type','apply-bulk-create',
        'apply-color-palette','apply-crop','apply-crop-page',
        'apply-document-style','apply-duration-to-all-scenes','apply-effects',
        'apply-filter','apply-font-recommendation','apply-generated-image',
        'apply-gradient-color','apply-library-asset','apply-loop-type',
        'apply-page-transition','apply-resize-page-variation','apply-scene-duration',
        'apply-shadow-effect','apply-shape-effect','apply-styles-to-all-pages',
        'apply-template-style','apply-text-effect','apply-text-effect-fit',
        'apply-text-effect-font','apply-text-outline','apply-to-all-font-rec',
        'apply-transition','apply-transition-to-all-pages',
        'apply-transition-to-all-scenes','apply-webpage-custom-theme',
        'apply-webpage-theme','auto-increase-scene-duration','change-page-name',
        'collapse-brand-colors','complete-av-upload','complete-remove-background',
        'complete-voice-recording','copy-scene','create-library',
        'create-new-file-from-template','create-webpage-custom-theme','cut-scene',
        'delete-page','detach-background','disable-text-fill','disable-text-outline',
        'duplicate-page','duplicate-resize-page','duplicate-scene',
        'duplicate-webpage-custom-theme','edit-slide-outline','edit-text-entity',
        'edit-webpage-custom-theme','edit-within-text-editor','expand-brand-colors',
        'fit-video','flip-content','generate-copywriter-assistant-result',
        'generate-image','generate-image-fill','generate-presentation',
        'generate-recommendation','generate-video','insert-webpage-button',
        'insert-webpage-element','insert-webpage-gif','insert-webpage-glideshow',
        'insert-webpage-photo','insert-webpage-photo-grid','insert-webpage-text',
        'modify-object-time','modify-volume-audio','modify-volume-video',
        'move-caption','move-content','mute-all-video','mute-video',
        'open-animation-panel','open-color-panel','open-insert-object-ai-panel',
        'open-remove-object-ai-panel','open-resize-panel','open-video-panel',
        'paste-scene','reorder-asset','reorder-scene','remove-all-animation',
        'remove-duration','remove-hyperlink','replace-asset','replace-background',
        'replace-linked-asset','replace-text','reset-genfill-image','resize-content',
        'resize-page','rotate-content','save-template','search-all','search-inspire',
        'select-add-page-transition','select-add-scene','select-add-transition',
        'select-align','select-asset','select-auto-enhance','select-bleed-toggle',
        'select-color','select-color-applied','select-color-eyedropper',
        'select-color-eyedropper-sample','select-color-format','select-color-palette',
        'select-copywriter-assistant-rewrite','select-create-collage','select-crop',
        'select-cutout-done','select-document-style','select-erase',
        'select-fit-to-content','select-font','select-generate-outline',
        'select-generate-slide','select-generative-fill',
        'select-generative-fill-upload','select-hyperlink',
        'select-insert-webpage-element','select-margins-toggle','select-page-styles',
        'select-page-visibility','select-remove-background','select-replace-asset',
        'select-restore-background','select-restore-brush','select-restore-image',
        'select-restore-video','select-social-safezone','select-task-within-resize',
        'select-template','select-text-effects','select-text-to-image',
        'select-translate-page','select-webpage-themes','set-as-background',
        'set-background-color','set-new-hyperlink','set-text-font-fallback',
        'slip-edit-video','split-scene','start-av-upload',
        'start-content-canvas-render','start-convert-to-presentation',
        'translate-page-complete','trim-audio','trim-scene','unmute-all-video',
        'unmute-video','upload-content-complete','upload-font','view-genai-results'
    )
    OR (lower(event_name) = 'select-remove-background'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'select-crop'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'apply-crop'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-insert-object-ai-panel'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-remove-object-ai-panel'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'select-erase'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'rotate-content'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-adjust-panel'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-effects-panel'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'adjust-opacity'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'flip-content'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-editor'
        AND lower(event_properties['custom.sdk.workflow_intent']) IN ('remove-background','crop-image')
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'select-crop-option'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'adjust-corner-radius'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'generate-image'
        AND lower(event_properties['custom.sdk.prompt_location']) != 'sdk-client-surface'
        AND lower(event_properties['custom.sdk.client_name']) LIKE '%acrobat%')
    OR (lower(event_name) = 'view-genai-results'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'apply-effects'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'apply-adjustments'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'select-cutout-done'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'scale-content'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'edit-text-entity'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'open-transform-panel'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'start-text-editor'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'resize-content'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'generate-image-fill'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'add-asset'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'content-transform-complete'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'change-zoom'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_subcategory) = 'edit'
        AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-desktop','acrobat-desktop-web'))
    OR (lower(event_name) = 'select-cover-page-service-start')
    OR (lower(event_name) = 'generate-presentation-complete')
    OR (lower(event_name) = 'view-image' AND lower(platform_category) = 'desktop-app')
    OR (lower(event_name) = 'export-project-complete' AND lower(platform_category) = 'desktop-app')
    OR (lower(event_name) = 'open-editor' AND lower(platform_category) = 'desktop-app')
    OR (lower(event_name) = 'select-template'
        AND lower(event_properties['custom.ui.location']) = 'explore'
        AND lower(event_properties['custom.referrer.app']) IN ('acrobat-desktop-studio','acrobat-web-studio'))
    OR (lower(event_properties['event.subcategory']) = 'edit'
        AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-dekstop','acrobat-desktop-web'))
    OR (lower(event_name) = 'publish-embed-project-complete'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'export-project-complete'
        AND lower(event_properties['custom.sdk.client_name']) = 'acrobat')
    OR (lower(event_name) = 'apply-filter'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'apply-auto-enhance'
        AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
    OR (lower(event_name) = 'export-project-complete'
        AND lower(event_properties['custom.referrer.app']) LIKE '%acrobat%')
"""

def build_kpi_engagement(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_ENGAGEMENT} AS
    WITH engaged_events AS (

      SELECT DISTINCT
        event_user_guid AS user_guid,
        event_dts,
        event_date
      FROM {T_TRON}
      WHERE event_date BETWEEN date_sub('{end_date}', 27) AND date_add('{end_date}', 1)
        AND auth_flag = true
        AND ({ENGAGED_EVENT_CONDITION_SQL})

      UNION

      SELECT DISTINCT
        event_visitor_guid AS user_guid,
        event_dts,
        event_date
      FROM {T_TRON}
      WHERE event_date BETWEEN date_sub('{end_date}', 27) AND date_add('{end_date}', 1)
        AND auth_flag = false
        AND ({ENGAGED_EVENT_CONDITION_SQL})
    )

    SELECT
      b.user_guid,

      MAX(CASE
        WHEN b.pnuv_flag = 1
         AND COALESCE(
              try_to_timestamp(e.event_dts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(e.event_dts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) BETWEEN COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             )
             AND COALESCE(
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
              try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
             ) + INTERVAL 1 DAY
        THEN 1 ELSE 0
      END) AS d1_engaged_flag,

      MAX(CASE
        WHEN to_date(e.event_date) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
        THEN 1 ELSE 0
      END) AS engaged_mau_flag

    FROM {T_FIRST_TS} b
    LEFT JOIN engaged_events e
      ON b.user_guid = e.user_guid
    GROUP BY b.user_guid
    """)


# ============================================================
# 9) KPI: All-user rWAU flags
# ============================================================

def build_kpi_rwau(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_RWAU} AS
    WITH weekly_active AS (
      SELECT DISTINCT
        ea_user_guid AS user_guid,
        CAST(FLOOR(
          datediff(to_date(ea_event_date), date_sub(to_date('{end_date}'), 27)) / 7
        ) AS INT) AS week_idx
      FROM {T_ACTIVITY}
      WHERE to_date(ea_event_date) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
    ),

    eligible_base AS (
      SELECT DISTINCT
        user_guid,
        week_idx AS base_week_idx
      FROM weekly_active
      -- Exclude the last week because its next-week return is outside the 28-day window.
      WHERE week_idx BETWEEN 0 AND 2
    ),

    returned_next_week AS (
      SELECT DISTINCT
        b.user_guid,
        b.base_week_idx
      FROM eligible_base b
      INNER JOIN weekly_active n
        ON b.user_guid = n.user_guid
       AND n.week_idx = b.base_week_idx + 1
    )

    SELECT
      b.user_guid,
      1 AS rwau_base_flag,
      MAX(CASE WHEN r.user_guid IS NOT NULL THEN 1 ELSE 0 END) AS rwau_return_flag
    FROM eligible_base b
    LEFT JOIN returned_next_week r
      ON b.user_guid = r.user_guid
     AND b.base_week_idx = r.base_week_idx
    GROUP BY b.user_guid
    """)


# ============================================================
# 10) User-level join table
# ============================================================

def build_user_level(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_USER_LEVEL} AS
    SELECT DISTINCT

      b.end_date,
      b.user_guid,
      b.platform_cut,
      b.new_or_return,

      CASE WHEN b.market_area NOT IN ('US', 'UK', 'IN', 'BRZ')
           THEN 'ROW' ELSE market_area
      END AS market_area,

      CASE
        WHEN COALESCE(rr.auth_flag,0) = 1 THEN 'authenticated'
        ELSE 'unauthenticated'
      END AS auth_status,

      COALESCE(rr.d1_auth_flag,0) AS d1_auth_flag,
      COALESCE(rr.auth_flag,0) AS mau_auth_flag,

      b.pnuv_flag,
      b.pnuv_wau,

      b.pnuv_t4_return_flag,
      b.mau_t4_return_flag,
      b.auth_flag AS nuv_auth_flag,

      b.pnuv_photos_flag,
      b.pnuv_photos_t4_return_flag,
      b.new_or_return_photos,

      b.core_flag,
      b.acrobat_photos_flag,
      b.acrobat_nonphotos_flag,
      b.express_photos_flag,
      b.standalone_excl_embed_edu_entitled_flag,
      b.edu_flag,
      b.cc_entitled_flag,
      b.third_party_flag,

      COALESCE(ed.d1_editor_open,0) AS d1_editor_open,
      COALESCE(ed.total_editor_open,0) AS total_editor_open,

      CASE WHEN COALESCE(ex.d1_export_cnt,0) >= 1 THEN 1 ELSE 0 END AS d1_export_flag,
      CASE WHEN COALESCE(ex.total_export_cnt,0) >= 1 THEN 1 ELSE 0 END AS total_export_flag,

      COALESCE(rr.d2_7_return_flag,0) AS d2_7_return_flag,
      COALESCE(rr.w1_return_flag,0) AS w1_return_flag,

      COALESCE(eng.d1_engaged_flag,0) AS d1_engaged_flag,
      COALESCE(eng.engaged_mau_flag,0) AS engaged_mau_flag,

      COALESCE(rw.rwau_base_flag,0) AS rwau_base_flag,
      COALESCE(rw.rwau_return_flag,0) AS rwau_return_flag

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_KPI_EDITOR} ed
      ON b.user_guid = ed.user_guid
    LEFT JOIN {T_KPI_EXPORT} ex
      ON b.user_guid = ex.user_guid
    LEFT JOIN {T_KPI_RETURNS} rr
      ON b.user_guid = rr.user_guid
    LEFT JOIN {T_KPI_ENGAGEMENT} eng
      ON b.user_guid = eng.user_guid
    LEFT JOIN {T_KPI_RWAU} rw
      ON b.user_guid = rw.user_guid
    """)


# ============================================================
# 11) NUV segment memberships and aggregation
# ============================================================

def build_nuf_segment_memberships_and_agg(end_date: str):

    sql_agg = f"""
    CREATE OR REPLACE TABLE {T_NUV_USER_LEVEL} AS
    WITH nuf_base AS (
      SELECT DISTINCT
        user_guid,
        CASE WHEN lower(a.platform_category) LIKE '%android%' THEN 'android'
             WHEN lower(a.platform_category) IN ('mobile-web','dapp-windows','desktop-web','ios','android') THEN platform_category
             ELSE 'desktop-web' END AS platform_category,
        CASE WHEN market_area NOT IN ('US', 'UK', 'IN', 'BRZ') THEN 'ROW'
             ELSE market_area END AS market_area,
        source_name,
        offer_category,
        channel_detail,
        b.cloud_type,
        market_segment_category,
        contract_type,
        offer_type
      FROM {T_NUF} a
      LEFT JOIN ms_categorization b
        ON a.category_key = offer_category_key
      WHERE date_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    ),

    nuf_overall_members AS (
      SELECT DISTINCT user_guid, platform_category platform_cut, market_area, NULL AS segment
      FROM nuf_base
    ),

    nuf_core_members AS (
      SELECT DISTINCT user_guid, platform_category platform_cut, market_area, 'Core' AS segment
      FROM nuf_base
      WHERE {CORE_FILTER_SQL}
    ),

    nuf_edu_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'edu' AS segment
      FROM nuf_base b
      JOIN ccex_tmp.edu_mau_on_date_tmp edu
        ON b.user_guid = edu.user_guid
       AND edu.as_of_date = '{end_date}'
    ),

    nuf_3p_members AS (
      SELECT DISTINCT user_guid, b.platform_category platform_cut, market_area, '3p' AS segment
      FROM nuf_base b
      LEFT JOIN event_activity_on_date_vw e
        ON b.user_guid = e.event_user_guid
      WHERE {third_party_filter_sql}
         OR airtel_offer_id = 1
    ),

    nuf_cc_entitled_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'cc entitled' AS segment
      FROM nuf_base b
      LEFT JOIN spark.cloud_type_mapping m
        ON b.cloud_type = m.cloud_type
      WHERE {CC_FILTER_SQL}
    ),

    nuf_photos_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'Acrobat photos' AS segment
      FROM nuf_base b
      JOIN event_activity_on_date_vw e
        ON b.user_guid = e.event_user_guid
      WHERE e.is_adobe_express_photos_user = true
    ),

    nuf_non_photos_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'Acrobat non-photos' AS segment
      FROM nuf_base b
      JOIN event_activity_on_date_vw e
        ON b.user_guid = e.event_user_guid
      WHERE is_acrobat_extension_user = true
         OR is_generative_reuse_user = true
         OR is_acrobat_mobile_edit_image_user = true
         OR is_enhanced_pdf_editing_user = true
         OR is_acrobat_original_creation_user = true
    ),

    nuf_express_photos_members AS (
      SELECT DISTINCT
        user_guid,
        CASE WHEN lower(platform_category) LIKE '%android%' THEN 'android'
             WHEN lower(platform_category) IN ('mobile-web','dapp-windows','desktop-web','ios','android') THEN platform_category
             ELSE 'desktop-web' END AS platform_cut,
        CASE WHEN market_area NOT IN ('US', 'UK', 'IN', 'BRZ') THEN 'ROW'
             ELSE market_area END AS market_area,
        'Express - Photos' AS segment
      FROM {T_MAU}
      WHERE date_date = '{end_date}'
        AND segment_tag = 'Express - Photos'
        AND TRIM(COALESCE(nuv_users,'')) <> ''
    ),

    nuf_standalone_product_members AS (
      SELECT DISTINCT
        user_guid,
        CASE WHEN lower(platform_category) LIKE '%android%' THEN 'android'
             WHEN lower(platform_category) IN ('mobile-web','dapp-windows','desktop-web','ios','android') THEN platform_category
             ELSE 'desktop-web' END AS platform_cut,
        CASE WHEN market_area NOT IN ('US', 'UK', 'IN', 'BRZ') THEN 'ROW'
             ELSE market_area END AS market_area,
        'Standalone(excl. Embed, excl. Edu Entitled)' AS segment
      FROM {T_MAU}
      WHERE date_date = '{end_date}'
        AND segment_tag = 'Product'
        AND TRIM(COALESCE(nuv_users,'')) <> ''
    ),

    nuf_memberships AS (
      SELECT * FROM nuf_overall_members
      UNION ALL SELECT * FROM nuf_core_members
      UNION ALL SELECT * FROM nuf_edu_members
      UNION ALL SELECT * FROM nuf_3p_members
      UNION ALL SELECT * FROM nuf_cc_entitled_members
      UNION ALL SELECT * FROM nuf_non_photos_members
      UNION ALL SELECT * FROM nuf_photos_members
      UNION ALL SELECT * FROM nuf_express_photos_members
      UNION ALL SELECT * FROM nuf_standalone_product_members
    ),

    nuv_agg AS (
      SELECT
        CASE WHEN GROUPING(segment)=1      THEN 'OVERALL' ELSE segment      END AS segment,
        CASE WHEN GROUPING(platform_cut)=1 THEN 'OVERALL' ELSE platform_cut END AS platform_cut,
        CASE WHEN GROUPING(market_area)=1  THEN 'OVERALL' ELSE market_area  END AS market_area,
        COUNT(DISTINCT user_guid) AS nuv_users
      FROM nuf_memberships
      GROUP BY GROUPING SETS (
        (),
        (segment),
        (platform_cut),
        (market_area),
        (segment, platform_cut),
        (segment, market_area),
        (platform_cut, market_area),
        (segment, platform_cut, market_area)
      )
    )

    SELECT * FROM nuv_agg
    """
    spark.sql(sql_agg)


# ============================================================
# 12) Write metrics
# ============================================================

def write_metrics(end_date: str):
    seg_sql_parts = []
    for s in SEGMENTS:
        seg_sql_parts.append(f"""
        SELECT '{s["name"]}' AS segment, *
        FROM {T_USER_LEVEL}
        WHERE {s["where"]}
        """)
    seg_union = "\nUNION ALL\n".join(seg_sql_parts)

    exec_sql(f"""
    INSERT OVERWRITE TABLE {T_OUT_METRICS}
    PARTITION (end_date = '{end_date}')

    WITH seg AS (
      {seg_union}
    ),

    mau_metrics AS (
      SELECT
        CASE WHEN GROUPING(platform_cut)=1  THEN 'OVERALL' ELSE platform_cut END AS platform_cut,
        CASE WHEN GROUPING(new_or_return)=1 THEN 'OVERALL' ELSE new_or_return END AS new_or_return,
        CASE WHEN GROUPING(market_area)=1   THEN 'OVERALL' ELSE market_area END AS market_area,
        CASE WHEN GROUPING(auth_status)=1   THEN 'OVERALL' ELSE auth_status END AS auth_status,
        segment,

        COUNT(DISTINCT user_guid) AS MAU,
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END) AS pnuv_users,
        CAST(NULL AS BIGINT) AS total_visitors,
        CAST(NULL AS BIGINT) AS nuv_users,

        -- New Users Funnel: D1 Auth
        COUNT(DISTINCT CASE
          WHEN pnuv_flag=1
           AND d1_auth_flag=1
          THEN user_guid
        END) AS d1_fta_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_flag=1
             AND d1_auth_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100,
        2) AS d1_fta_pct,

        -- All Users Funnel: Auth Rate
        COUNT(DISTINCT CASE
          WHEN mau_auth_flag=1
          THEN user_guid
        END) AS fta_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN mau_auth_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100,
        2) AS fta_pct,

        -- New Users Funnel: D1 Editor
        COUNT(DISTINCT CASE
          WHEN pnuv_flag=1
           AND d1_editor_open=1
          THEN user_guid
        END) AS d1_editor_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_flag=1
             AND d1_editor_open=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100,
        2) AS d1_editor_pct,

        -- New Users Funnel: D1 Engaged
        COUNT(DISTINCT CASE
          WHEN pnuv_flag=1
           AND d1_engaged_flag=1
          THEN user_guid
        END) AS d1_engaged_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_flag=1
             AND d1_engaged_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100,
        2) AS d1_engaged_pct,

        -- All Users Funnel: Editor Rate
        COUNT(DISTINCT CASE
          WHEN total_editor_open=1
          THEN user_guid
        END) AS editor_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN total_editor_open=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100,
        2) AS editor_pct,

        -- New Users Funnel: D1 Export
        COUNT(DISTINCT CASE
          WHEN pnuv_flag=1
           AND d1_export_flag=1
          THEN user_guid
        END) AS d1_export_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_flag=1
             AND d1_export_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100,
        2) AS d1_export_pct,

        -- All Users Funnel: Export Rate
        COUNT(DISTINCT CASE
          WHEN total_export_flag=1
          THEN user_guid
        END) AS export_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN total_export_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100,
        2) AS export_pct,

        -- New Users Funnel: D2-D7 Return
        COUNT(DISTINCT CASE
          WHEN pnuv_flag=1
           AND d2_7_return_flag=1
          THEN user_guid
        END) AS d2_7_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_flag=1
             AND d2_7_return_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100,
        2) AS d2_7_pct,

        -- New Users Funnel: W1 Return
        COUNT(DISTINCT CASE
          WHEN pnuv_wau=1
          THEN user_guid
        END) AS pnuv_wau_users,

        COUNT(DISTINCT CASE
          WHEN pnuv_wau=1
           AND w1_return_flag=1
          THEN user_guid
        END) AS w1_rr_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN pnuv_wau=1
             AND w1_return_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_wau=1 THEN user_guid END), 0) * 100,
        2) AS w1_rr_pct,

        -- New Users Funnel: M1RR numerator
        COUNT(DISTINCT CASE
          WHEN acrobat_photos_flag=1
           AND pnuv_photos_flag=1
           AND pnuv_photos_t4_return_flag=1
          THEN user_guid

          WHEN acrobat_photos_flag=0
           AND pnuv_flag=1
           AND pnuv_t4_return_flag=1
          THEN user_guid
        END) AS m1_rr_users,

        -- New Users Funnel: M1RR base
        COUNT(DISTINCT CASE
          WHEN acrobat_photos_flag=1
           AND pnuv_photos_flag=1
          THEN user_guid

          WHEN acrobat_photos_flag=0
           AND pnuv_flag=1
          THEN user_guid
        END) AS m1_rr_base_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN acrobat_photos_flag=1
             AND pnuv_photos_flag=1
             AND pnuv_photos_t4_return_flag=1
            THEN user_guid

            WHEN acrobat_photos_flag=0
             AND pnuv_flag=1
             AND pnuv_t4_return_flag=1
            THEN user_guid
          END)
          / NULLIF(
              COUNT(DISTINCT CASE
                WHEN acrobat_photos_flag=1
                 AND pnuv_photos_flag=1
                THEN user_guid

                WHEN acrobat_photos_flag=0
                 AND pnuv_flag=1
                THEN user_guid
              END), 0
            ) * 100,
        2) AS m1_rr_pct,

        -- All Users Funnel: rWAU
        COUNT(DISTINCT CASE
          WHEN rwau_base_flag=1
          THEN user_guid
        END) AS rwau_base_users,

        COUNT(DISTINCT CASE
          WHEN rwau_base_flag=1
           AND rwau_return_flag=1
          THEN user_guid
        END) AS rwau_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN rwau_base_flag=1
             AND rwau_return_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT CASE WHEN rwau_base_flag=1 THEN user_guid END), 0) * 100,
        2) AS rwau_pct,

        -- All Users Funnel: rMAU
        COUNT(DISTINCT CASE
          WHEN mau_t4_return_flag=1
          THEN user_guid
        END) AS rmau_users,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN mau_t4_return_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100,
        2) AS rmau_pct,

        -- All Users Funnel: Engaged Rate
        COUNT(DISTINCT CASE
          WHEN engaged_mau_flag=1
          THEN user_guid
        END) AS engaged_mau,

        ROUND(
          COUNT(DISTINCT CASE
            WHEN engaged_mau_flag=1
            THEN user_guid
          END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100,
        2) AS engaged_mau_pct

      FROM seg
      GROUP BY segment,
        GROUPING SETS (
          (),
          (platform_cut),
          (new_or_return),
          (market_area),
          (auth_status)
        )
    )

    SELECT
      m.platform_cut,
      m.new_or_return,
      m.market_area,
      m.auth_status,
      m.segment,
      m.MAU,
      m.pnuv_users,

      CASE
        WHEN m.new_or_return='OVERALL' AND m.auth_status='OVERALL'
        THEN n.nuv_users
        ELSE NULL
      END AS nuv_users,

      m.total_visitors,
      m.d1_fta_users,
      m.d1_fta_pct,
      m.fta_users,
      m.fta_pct,
      m.d1_editor_users,
      m.d1_editor_pct,
      m.editor_users,
      m.editor_pct,
      m.d1_export_users,
      m.d1_export_pct,
      m.export_users,
      m.export_pct,
      m.d2_7_users,
      m.d2_7_pct,
      m.w1_rr_users,
      m.w1_rr_pct,
      m.m1_rr_users,
      m.m1_rr_pct,
      m.rmau_users,
      m.rmau_pct,
      m.engaged_mau,
      m.engaged_mau_pct,
      m.d1_engaged_users,
      m.d1_engaged_pct,
      m.rwau_users,
      m.rwau_pct,
      m.pnuv_wau_users,
      m.m1_rr_base_users,
      m.rwau_base_users

    FROM mau_metrics m
    LEFT JOIN {T_NUV_USER_LEVEL} n
      ON m.platform_cut = n.platform_cut
     AND m.segment = n.segment
     AND m.market_area = n.market_area
    """)

# ============================================================
# 13) Cleanup
# ============================================================

def drop_all_temps():
    for t in [
        T_BASE,
        T_FIRST_TS,
        T_ACTIVITY,
        T_EXPORT_ACT,
        T_KPI_EDITOR,
        T_KPI_EXPORT,
        T_KPI_RETURNS,
        T_KPI_ENGAGEMENT,
        T_KPI_RWAU,
        T_USER_LEVEL
    ]:
        exec_sql(f"DROP TABLE IF EXISTS {t}")

In [0]:
build_base_dedup('2026-05-22')
build_first_ts('2026-05-22')
build_activity('2026-05-22')
build_export_activity('2026-05-22')
build_kpi_editor('2026-05-22')
build_kpi_export('2026-05-22')
build_kpi_returns('2026-05-22')
build_kpi_engagement('2026-05-22')
build_kpi_rwau('2026-05-22')
build_user_level('2026-05-22')
build_event_activity_view('2026-05-22')
build_nuf_segment_memberships_and_agg('2026-05-22')
write_metrics('2026-05-22')

In [0]:
%sql
SELECT
  segment,
  MAU,
  pnuv_users,
  nuv_users,

  d1_fta_users,
  d1_fta_pct,
  d1_editor_users,
  d1_editor_pct,
  d1_engaged_users,
  d1_engaged_pct,
  d1_export_users,
  d1_export_pct,

  w1_rr_users,
  pnuv_wau_users,
  w1_rr_pct,

  m1_rr_users,
  m1_rr_base_users,
  m1_rr_pct,

  fta_users,
  fta_pct,
  editor_users,
  editor_pct,
  engaged_mau,
  engaged_mau_pct,
  export_users,
  export_pct,

  rwau_users,
  rwau_base_users,
  rwau_pct,

  rmau_users,
  rmau_pct
FROM ccex_tmp.core_leading_indicator_metrics_mau_v2
WHERE end_date = '2026-05-22'
  AND platform_cut = 'OVERALL'
  AND new_or_return = 'OVERALL'
  AND market_area = 'OVERALL'
  AND auth_status = 'OVERALL'
ORDER BY segment;

segment,MAU,pnuv_users,nuv_users,d1_fta_users,d1_fta_pct,d1_editor_users,d1_editor_pct,d1_engaged_users,d1_engaged_pct,d1_export_users,d1_export_pct,w1_rr_users,pnuv_wau_users,w1_rr_pct,m1_rr_users,m1_rr_base_users,m1_rr_pct,fta_users,fta_pct,editor_users,editor_pct,engaged_mau,engaged_mau_pct,export_users,export_pct,rwau_users,rwau_base_users,rwau_pct,rmau_users,rmau_pct
3p,1872511,555963,523187,378637,68.1,269331,48.44,240976,43.34,72281,13.0,21986,136053,16.16,0,557843,0.0,1187772,63.43,1008711,53.87,883322,47.17,384630,20.54,295782,1114728,26.53,0,0.0
Acrobat non-photos,19544194,14462858,19840912,2782700,19.24,2698671,18.66,1131240,7.82,686513,4.75,262451,4273475,6.14,0,14474332,0.0,7839324,40.11,19218272,98.33,8050094,41.19,4391795,22.47,1819550,14800204,12.29,0,0.0
Acrobat photos,8565183,7320993,7170253,278592,3.81,69849,0.95,57737,0.79,85266,1.16,1928582,5769417,33.43,0,2620108,0.0,1548932,18.08,2020699,23.59,1416069,16.53,3889796,45.41,3696391,7256285,50.94,0,0.0
Express - Photos,8794120,7359023,7359023,301039,4.09,79295,1.08,66232,0.9,92401,1.26,1932164,5780302,33.43,0,2658138,0.0,1764023,20.06,2117583,24.08,1495266,17.0,4005380,45.55,3789110,7438424,50.94,0,0.0
OVERALL,50625530,37018492,61468529,6550181,17.69,5019708,13.56,3256571,8.8,1827259,4.94,2440182,14012338,17.41,0,32317607,0.0,19571288,38.66,35215467,69.56,20263855,40.03,13864383,27.39,7849055,39117452,20.07,0,0.0
"Standalone(excl. Embed, excl. Edu Entitled)",20212332,14706470,14706470,2968794,20.19,2110184,14.35,1627185,11.06,787379,5.35,283624,3864049,7.34,0,14684623,0.0,8439036,41.75,13325955,65.93,9305788,46.04,4587276,22.7,2447624,15801400,15.49,0,0.0
cc entitled,2317340,382765,394129,359545,93.93,228890,59.8,152235,39.77,79560,20.79,8075,94225,8.57,0,398723,0.0,2244091,96.84,1738847,75.04,1337989,57.74,992300,42.82,581373,1847007,31.48,0,0.0
edu,4231507,1480053,1505777,1163195,78.59,657945,44.45,745345,50.36,453961,30.67,24515,316347,7.75,0,1488219,0.0,3920271,92.64,2684954,63.45,2784430,65.8,2001380,47.3,714123,3499696,20.41,0,0.0


In [0]:
START_END_DATE = "2025-11-28"
END_END_DATE   = "2026-05-22"
STEP_DAYS      = 7

for end_date in daterange(START_END_DATE, END_END_DATE, STEP_DAYS):
    print(f"Running MBR pipeline for {end_date}")

    build_base_dedup(end_date)
    build_first_ts(end_date)
    build_activity(end_date)
    build_export_activity(end_date)

    build_kpi_editor(end_date)
    build_kpi_export(end_date)
    build_kpi_returns(end_date)
    build_kpi_engagement(end_date)
    build_kpi_rwau(end_date)

    build_user_level(end_date)

    build_event_activity_view(end_date)
    build_nuf_segment_memberships_and_agg(end_date)

    write_metrics(end_date)

    print(f"Completed MBR pipeline for {end_date}")

Running MBR pipeline for 2025-11-28
Completed MBR pipeline for 2025-11-28
Running MBR pipeline for 2025-12-05
Completed MBR pipeline for 2025-12-05
Running MBR pipeline for 2025-12-12
Completed MBR pipeline for 2025-12-12
Running MBR pipeline for 2025-12-19
Completed MBR pipeline for 2025-12-19
Running MBR pipeline for 2025-12-26
Completed MBR pipeline for 2025-12-26
Running MBR pipeline for 2026-01-02
Completed MBR pipeline for 2026-01-02
Running MBR pipeline for 2026-01-09
Completed MBR pipeline for 2026-01-09
Running MBR pipeline for 2026-01-16
Completed MBR pipeline for 2026-01-16
Running MBR pipeline for 2026-01-23
Completed MBR pipeline for 2026-01-23
Running MBR pipeline for 2026-01-30
Completed MBR pipeline for 2026-01-30
Running MBR pipeline for 2026-02-06
Completed MBR pipeline for 2026-02-06
Running MBR pipeline for 2026-02-13
Completed MBR pipeline for 2026-02-13
Running MBR pipeline for 2026-02-20
Completed MBR pipeline for 2026-02-20
Running MBR pipeline for 2026-02-27
Co

In [0]:
%sql
Select count(distinct user_guid)
from ccex_tmp.ccex_active_user_temp_ch_tmp_mau
where end_date='2026-05-22';

count(DISTINCTuser_guid)
50113635


In [0]:
%sql
select * from ccex_tmp.temp_okr_cube_Data_v3 limit 10;

okr_segment,okr_subsegment,layer,segment_tag,value,fiscal_yr,fiscal_yr_and_qtr_desc,fiscal_wk_in_yr,fiscal_wk_in_qtr,fiscal_wk_starting_date,fiscal_wk_ending_date,kpi,platform_category,market_area,offer_category,entry_source,source_name,new_or_return,grp,insert_ts,date_date,key,numerator,denominator,m1rr,fiscal_year,fiscal_qtr,fiscal_week_in_qtr,fiscal_year_and_qtr_desc,pqe_fiscal_yr,pqe_fiscal_qtr,value_pqe,numerator_pqe,denominator_pqe,m1rr_pqe,value_pw,numerator_pw,denominator_pw,m1rr_pw,target,qrf,outlook,year_target
Standalone,null,null,Standalone,2846,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,desktop-web,FRA,CC Free,A.com/express-Create,CCEX,NEW,0000000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,70,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,android,ANZ,Team,Mobile,OVERALL,RETURN,0010000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,4,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,desktop-web,KOR,Edu D2S,Edu,CCEX,RETURN,0000000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,2,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,desktop-web,RCIS,Enterprise ETLA,First Party - CCH,CCEX,RETURN,0000000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,4,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,OVERALL,NORD,Team,A.com/express-Rest,OVERALL,OVERALL,0110001,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,5,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,desktop-web,UK,Team,Acrobat - Extension,CCEX,OVERALL,0100000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,12,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,OVERALL,SWI,Edu Enterprise K12,Edu,CCEX,OVERALL,0100001,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,55,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,desktop-web,ME,Team,First Party - CCH,CCEX,NEW,0000000,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,1022,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,OVERALL,BRZ,Trial,D2P,CCEX,RETURN,0000001,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
Standalone,null,null,Standalone,15,2025,2025-Q4,52,13,2025-11-22,2025-11-28,DDOM Total MAU GA,OVERALL,US,Team,A.com/express-Homepage,ESDK,RETURN,0000001,2026-05-20T22:01:30.160Z,2025-11-28,DDOM_TOTAL_MAU_GA,null,null,null,2025,4,13,2025-Q4,2025,3,null,null,null,null,null,null,null,null,null,null,null,null
